# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "iron"
vehicle = "salt"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.019944
                            2                salt            0.019937
                            3                salt            0.019941
                            4                salt            0.019950
                            5                salt            0.019950
        5          15       1                salt            0.019927
                            2                salt            0.019949
                            3                salt            0.019941
                            4                salt            0.019949
                            5                salt            0.019946
        15         30       1                salt            0.019941
                            2                salt            0.019942
                            3                salt            0.019947
                            4   

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.019944
                            2                salt            0.019937
                            3                salt            0.019941
                            4                salt            0.019950
                            5                salt            0.019950
        5          15       1                salt            0.019927
                            2                salt            0.019949
                            3                salt            0.019941
                            4                salt            0.019949
                            5                salt            0.019946
        15         30       1                salt            0.019941
                            2                salt            0.019942
                            3                salt            0.019947
                            4   

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

wealth_quintile  vehicle_name
1                salt            0
2                salt            0
3                salt            0
4                salt            0
5                salt            0
Name: value, dtype: int64

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.019944
                            2                salt            0.019937
                            3                salt            0.019941
                            4                salt            0.019950
                            5                salt            0.019950
        5          15       1                salt            0.019927
                            2                salt            0.019949
                            3                salt            0.019941
                            4                salt            0.019949
                            5                salt            0.019946
        15         30       1                salt            0.019941
                            2                salt            0.019942
                            3                salt            0.019947
                            4   

In [9]:
from lsff_utils import config_utils

scenarios = config_utils.get_intervention_scenarios(location, vehicle)
scenarios

['intervention']

In [10]:
any_consumption = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
any_consumption = any_consumption.set_index(
    [c for c in any_consumption.columns if c != "value"]
).value
any_consumption

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.902333
                            2                salt            0.890204
                            3                salt            0.883991
                            4                salt            0.891527
                            5                salt            0.873610
        5          15       1                salt            0.974985
                            2                salt            0.996484
                            3                salt            1.000000
                            4                salt            1.000000
                            5                salt            1.000000
        15         30       1                salt            0.988071
                            2                salt            0.986670
                            3                salt            0.998649
                            4   

In [11]:
if (
    len(
        any_consumption.reset_index()[["sex", "age_start", "age_end"]].drop_duplicates()
    )
    == 1
):
    # Assumed same for all ages/sexes
    any_consumption = any_consumption.droplevel(["sex", "age_start", "age_end"])
    display(any_consumption)

In [12]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.331736
                            2                salt            0.331627
                            3                salt            0.331695
                            4                salt            0.331843
                            5                salt            0.331843
        5          15       1                salt            0.331461
                            2                salt            0.331821
                            3                salt            0.331700
                            4                salt            0.331821
                            5                salt            0.331785
        15         30       1                salt            0.331702
                            2                salt            0.331709
                            3                salt            0.331787
                            4   

In [13]:
import pathlib, numpy as np

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (
        (target_coverage > current_coverage.reindex_like(target_coverage))
        | np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ).all()
    target_coverage[
        np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ] = current_coverage.reindex_like(target_coverage)
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = any_consumption * target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.305197
                            2                salt            0.305097
                            3                salt            0.305159
                            4                salt            0.305296
                            5                salt            0.305296
        5          15       1                salt            0.304944
                            2                salt            0.305275
                            3                salt            0.305164
                            4                salt            0.305275
                            5                salt            0.305242
        15         30       1                salt            0.305166
                            2                salt            0.305172
                            3                salt            0.305244
                            4   

In [14]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [15]:
effective_baseline_coverage = any_consumption * current_coverage * effectiveness
effective_baseline_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                salt            0.010258
        5          15       1                salt            0.011074
        15         30       1                salt            0.011231
        30         50       1                salt            0.011368
        50         125      1                salt            0.011353
Male    0          5        1                salt            0.010304
        5          15       1                salt            0.011216
        15         30       1                salt            0.011231
        30         50       1                salt            0.011360
        50         125      1                salt            0.011356
Female  0          5        2                salt            0.010116
        5          15       2                salt            0.011331
        15         30       2                salt            0.011215
        30         50       2   

In [16]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)